# Notebook 01 — Baseline MedGemma

**Mastercamp EFREI 2025-2026 — Assistant Radiologue Virtuel**

Objectif : tester le prompt baseline sur les 20 images de smoke test et calculer les premières métriques.

> ⚠️ Ce notebook ne produit aucun diagnostic médical. Prototype pédagogique uniquement.

## 1. Imports et configuration

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import csv
import pandas as pd
from pathlib import Path
from PIL import Image

from src.inference import run_inference, run_inference_toy
from src.guardrails import validate_and_apply
from src.database import log_prediction, init_db
from src.metrics import compute_metrics, print_metrics
from src.preprocessing import load_and_preprocess

DATA_DIR = Path('../data/sample_images')
PROMPT_PATH = Path('../prompts/prompt_baseline.txt')
DB_PATH = Path('../data/baseline_evidence.sqlite')
OUT_DIR = Path('../eval/outputs/baseline')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Configuration OK')

## 2. Chargement des images et labels

Le fichier `labels.csv` doit être présent dans `data/sample_images/` avec les colonnes `image_name` et `ground_truth`.

In [ ]:
label_file = DATA_DIR / 'labels.csv'

if label_file.exists():
    labels_df = pd.read_csv(label_file)
    labels = dict(zip(labels_df['image_name'], labels_df['ground_truth']))
    image_files = sorted(DATA_DIR.glob('*.png')) + sorted(DATA_DIR.glob('*.jpg'))
    print(f'{len(image_files)} images trouvées, {len(labels)} labels chargés')
    print(labels_df['ground_truth'].value_counts())
else:
    print('labels.csv introuvable — mode toy activé')
    labels = {}
    image_files = []

## 3. Inférence baseline

On envoie chaque image à MedGemma avec le prompt baseline et on enregistre chaque résultat.

In [ ]:
init_db(DB_PATH)
results = []

# Si pas d'images réelles, on utilise le mode toy
if not image_files:
    print('Mode toy — 20 cas synthétiques')
    toy_labels = ['normal']*7 + ['suspected_opacity']*7 + ['uncertain']*6
    for i, gt in enumerate(toy_labels):
        name = f'CXR_SYN_{i+1:03d}_{gt}.png'
        raw, latency = run_inference_toy(name)
        result = validate_and_apply(raw)
        log_prediction(name, 'baseline_v1', 'toy_model', result, latency, DB_PATH)
        results.append({'image_name': name, 'ground_truth': gt,
                        'predicted_class': result['predicted_class'],
                        'confidence': result['confidence'],
                        'image_quality': result['image_quality'],
                        'latency_ms': latency, 'json_valid': 1, 'hallucination': 0})
        print(f"  {'✓' if result['predicted_class']==gt else '✗'} {name} → {result['predicted_class']} ({result['confidence']})")
else:
    for img_path in image_files:
        image = load_and_preprocess(img_path)
        raw, latency = run_inference(image, PROMPT_PATH)
        try:
            result = validate_and_apply(raw)
            json_valid = 1
        except ValueError as e:
            print(f'  ⚠ {img_path.name} : {e}')
            result = {'predicted_class': 'uncertain', 'confidence': 0.0, 'warning': 'JSON invalide'}
            json_valid = 0
        gt = labels.get(img_path.name, 'unknown')
        log_prediction(img_path.name, 'baseline_v1', 'medgemma-4b', result, latency, DB_PATH)
        results.append({'image_name': img_path.name, 'ground_truth': gt,
                        'predicted_class': result.get('predicted_class'),
                        'confidence': result.get('confidence'),
                        'image_quality': result.get('image_quality'),
                        'latency_ms': latency, 'json_valid': json_valid, 'hallucination': 0})
        print(f"  {'✓' if result.get('predicted_class')==gt else '✗'} {img_path.name} → {result.get('predicted_class')} ({result.get('confidence')})")

print(f'\n{len(results)} cas traités')

## 4. Métriques baseline

In [ ]:
df = pd.DataFrame(results)
display(df)

valid_results = [r for r in results if r['ground_truth'] not in ('unknown',)]
metrics = compute_metrics(valid_results)
print_metrics(metrics)

# Sauvegarde CSV
df.to_csv(OUT_DIR / 'baseline_results.csv', index=False)
print(f'Résultats sauvegardés : {OUT_DIR / "baseline_results.csv"}')

## 5. Distribution des prédictions

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution des classes prédites
df['predicted_class'].value_counts().plot(kind='bar', ax=axes[0], color=['#1d9e75','#e85d24','#7f77dd'])
axes[0].set_title('Distribution des classes prédites (baseline)')
axes[0].set_xlabel('Classe')
axes[0].set_ylabel('Nombre de cas')
axes[0].tick_params(rotation=0)

# Distribution des scores de confiance
df['confidence'].hist(bins=10, ax=axes[1], color='#185fa5', edgecolor='white')
axes[1].axvline(x=0.60, color='red', linestyle='--', label='Seuil garde-fou (0.60)')
axes[1].set_title('Distribution des scores de confiance')
axes[1].set_xlabel('Confiance')
axes[1].set_ylabel('Fréquence')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / 'baseline_distributions.png', dpi=150)
plt.show()
print('Graphiques sauvegardés')

## 6. Résumé des limites observées

Compléter manuellement cette section après analyse des résultats.

In [ ]:
print('=== RÉSUMÉ BASELINE ===')
print(f"Accuracy      : {metrics.get('accuracy', 'N/A')}")
print(f"Macro-F1      : {metrics.get('macro_f1', 'N/A')}")
print(f"JSON valide   : {metrics.get('json_valid_rate', 'N/A')}")
print(f"Incertitude   : {metrics.get('uncertainty_rate', 'N/A')}")
print(f"Latence moy.  : {metrics.get('avg_latency_ms', 'N/A')} ms")
print()
print('Limites à documenter :')
print('  - [à compléter après analyse manuelle des erreurs]')
print('  - [à compléter après analyse manuelle des erreurs]')